### nxsut generator — v3.1

MARIO-native pipeline: parses EXIOBASE Hybrid v3.3.18, adds the explicit steel & H2 routes + furnace-gas reallocation, updates the **electricity** supply mix from EMBER and the **steel** supply mix (primary/secondary routes) from worldsteel, pools **electricity, steel and aluminium** trade behind a supply/need pass-through layer, and updates the trade mixes — electricity from the open **ENTSO-E** scheduled exchanges, steel/aluminium (pooled) and chemicals / N-fertiliser / non-metallic minerals / non-ferrous metals (Isard) from **BACI (CEPII)** bilateral flows.

Pooling vs Isard is decided by the pooling KPI (materiality × footprint heterogeneity over physically-traded goods): the headline materials (electricity, steel, aluminium) get a first-class `… need` commodity; the rest have only their import origins rewritten (`level='Commodity'`, zero dimension cost). Energy carriers stay aggregated (CN can't separate refined-petroleum products by product).

**Data access.** EMBER (supply mix) and ENTSO-E (electricity trade) come from the **nxbase query API** — no local files. The only large *local* inputs are the **EXIOBASE Hybrid** base table (`raw`) and the **BACI** trade files (required for the material trade mixes): BACI is heavy so it is **not hosted** on the public nxbase (`visibility='local'`), and `get_trade_mix` falls back to the **raw CEPII BACI files** (set `baci` in `paths.yml`; see the download cell below) when the API has no BACI — verified identical to the API path, and the file covers *all* chapters. So v3.1 is reproducible from the public open API + those two local files. The one remaining local-only nxbase source is the small **worldsteel** steel supply mix (`WSTEEL`) — promote it to `open`, or run the steel supply step against a local nxbase.

Configure your paths.yml file, set `user` and `year` in the first cell, then run top to bottom.

In [ ]:
import mario
import yaml
import os

pfile = 'paths_personal.yml' if os.path.exists('paths_personal.yml') else 'paths.yml'
with open(pfile, 'r') as file:  # personal override (git-ignored), else the template
    paths = yaml.safe_load(file)

user = 'USER'   # your key in paths.yml
# defaults for interactive use; a headless run overrides them via env
year = int(os.environ.get('NXSUT_YEAR', 2025))   # the year you want to build
version = os.environ.get('NXSUT_VERSION', 'v3.2')   # v3.2 = v3.1 + transport layer
print(f'building nxsut {version} for {year}')

paths = paths[user]
import warnings
warnings.filterwarnings("ignore")

from support import nxbase_client as nxc
nxbase_api = paths.get('nxbase_api', nxc.DEFAULT_API)


#### BACI trade files — required for the material trade mixes

The steel / aluminium (pooled) and chemicals / N-fertiliser / non-metallic / non-ferrous (Isard) trade mixes **require** **BACI (CEPII)** bilateral trade. BACI is *not* hosted on the public nxbase (too large), so — until it is — you must supply it locally (a one-off manual download, like the EXIOBASE Hybrid base table) **unless** you build against a local nxbase that already ingested BACI:

1. Download the **HS 2022** release from **[CEPII BACI](https://www.cepii.fr/CEPII/en/bdd_modele/bdd_modele_item.asp?id=37)** (free registration; DOI `10.15454/HL2VHT`).
2. Unzip it and set **`baci`** in `paths.yml` to that folder — it must contain `BACI_HS22_Y<year>_V*.csv` and `country_codes_*.csv`.

`get_trade_mix` then reads those files directly (M49→ISO2 from BACI's own `country_codes`), identical to the nxbase path. If `baci` is unset **and** the nxbase API has no BACI, the trade-mix cell raises. Building against a **local** nxbase that already ingested BACI needs none of this.

Parse the raw EXIOBASE database, aggregate electricity to EMBER resolution, then add the explicit steel & H2 production routes (Ghezzi et al. 2026 recipe, fetched from the nxbase query API). `add_sectors` runs **after** `aggregate_ee`: the routes attach to the single aggregated grid `Electricity`, and the pre-existing EMBER electricity activities keep their supply coefficients. `meta.source` enables MARIO's EXIOBASE Rest-of-World member-country expansion when using EMBER.

In [ ]:
db = mario.parse_from_txt(paths['raw'], table='SUT', mode='flows')
db.meta.source = 'EXIOBASE Hybrid 3.3.18'
db.aggregate('support/aggregate_ee.xlsx', ignore_nan=True)

# steel/H2 sectors from the nxbase recipe -- add AFTER aggregate_ee so the new
# routes attach to the aggregated grid 'Electricity', and the pre-existing EMBER
# electricity activities keep their supply coefficients (running add_sectors
# before aggregate would drop them from the 's' block -> update_supply_mix fails).
nxc.build_add_sectors_master('support/add_sectors/Master_steel_h2.xlsx', '_steel_master.xlsx', api_url=nxbase_api)
db.read_add_sectors_excel('_steel_master.xlsx', read_inventories=True)
db.add_sectors()

### Furnace-gas emission reallocation (ExioSteel method)

Two fictitious activities take the blast/oxygen furnace gas by-products (the steel sector's supply of them is zeroed), and the steel sector's coefficients are recomputed on its steel supply alone (`U/S_main`, not `U/X`) → footprint per tonne of *actual steel*, not diluted by the co-product gases. Runs **after** the Ghezzi `add_sectors`, **before** the supply mix.

In [ ]:
# Furnace-gas emission reallocation (ExioSteel method): two fictitious gas-
# production activities take the blast/oxygen furnace gas by-products (steel
# supply of them zeroed), and the steel sector's U/V/E are recomputed on its
# steel supply alone (/ S_main, not / X) -> footprint per tonne of actual steel.
db.read_add_sectors_excel('support/add_sectors/blastfurnacegas.xlsx', read_inventories=True)
db.add_sectors()

STEEL_ACT = 'Manufacture of basic iron and steel and of ferro-alloys and first products thereof'
STEEL_COM = 'Basic iron and steel and of ferro-alloys and first products thereof'
s, u, v, e = db.s, db.u, db.v, db.e
by_product = list(db.add_sectors_master['Commodity'].unique())
gas_of_act = {a: db.add_sectors_master.loc[db.add_sectors_master['Activity'] == a, 'Commodity'].values[0]
              for a in db.new_activities}
for region in db.get_index('Region'):
    s_byprod = (s.loc[(region, 'Activity', STEEL_ACT), (region, 'Commodity', by_product)] * 0).to_frame().T
    s.update(s_byprod)
    for new_act, commodity in gas_of_act.items():
        s.loc[(region, 'Activity', new_act), (region, 'Commodity', commodity)] = 1
    S_main = db.S.loc[(region, 'Activity', STEEL_ACT), (region, 'Commodity', STEEL_COM)]
    u.update(db.U.loc[:, (region, 'Activity', STEEL_ACT)] / S_main.sum())
    v.update(db.V.loc[:, (region, 'Activity', STEEL_ACT)] / S_main.sum())
    e.update(db.E.loc[:, (region, 'Activity', STEEL_ACT)] / S_main.sum())

z = db.z
z.update(s)
z.update(u)
db.update_scenarios('baseline', z=z, v=v, e=e)
db.reset_to_coefficients('baseline')
print('BFG/OFG reallocation applied; new activities:', list(db.new_activities))

### Transport service layer — Moves B + A + C

Disaggregates transport before the mix updates (design: `docs/transport_service_layer_plan.md`):
**B** splits the commercial transport sectors into freight/passenger children re-denominated
to observed tkm/pkm; **A** turns private mobility into household-operated vehicle activities
producing a `Private road mobility` commodity (household fuels rerouted from Y, tailpipe
re-attributed EY→E); **C** externalises own-account road freight into its own activity and
commodity — which aligns the SUT perimeters with the UNSD balances (all road fuel sits in
transport-family columns; industry rows keep process, heating and off-road).

Runs here so that the new electricity input of the BEV activity joins the electricity pooling
downstream. Reproduction inputs (specs, keys, provenance) live in `transport/data/`.


In [ ]:
from transport.pipeline import apply_transport_layer

apply_transport_layer(db)


Supply mix from nxbase (query API) — see `support/nxbase_client.py`. MARIO reads the reduced EMBER snapshot from a transient file, regenerated every run.

In [ ]:
from support import nxbase_client as nxc

nxbase_api = paths.get('nxbase_api', nxc.DEFAULT_API)
print(nxc.get_provenance(nxbase_api, [
    'EMBER Yearly Electricity Data (generation) 2025',
    'UNSD Energy Statistics — electricity & heat production 2021-2023',
]))

# UNSD-first, EMBER-as-arbiter (step 0): per country the generation mix comes
# from UNSD.GEN where the arbitrated selection says so (CHP, heat and
# autoproducers included), EMBER everywhere else and for years UNSD does not
# cover. Same 4-column shape MARIO reads natively.
ember_snapshot_path = 'support/_nxbase_ember_snapshot.csv'
nxc.get_supply_mix_snapshot(nxbase_api, years=(year,)).to_csv(
    ember_snapshot_path, index=False)

db.update_supply_mix(
    "electricity",
    scenario = 'baseline',
    year = year,
    ember_path = ember_snapshot_path,
)

In [ ]:
# Steel primary/secondary supply mix (worldsteel, governed in nxbase as WSTEEL),
# applied before pooling like the electricity supply mix: rewrites the
# BF-BOF+DRI-EAF (primary) vs scrap-EAF (secondary) market shares of
# 'Basic iron and steel' per region. WSTEEL is visibility='local' -> needs the
# local nxbase API (nxbase_api pointing at the local instance).
steel_supply = nxc.get_steel_supply_mix(nxbase_api, year, regions=list(db.get_index('Region')))
db.update_supply_mix(steel_supply, level='Activity',
                     commodities=[nxc.STEEL_COMMODITY], scenario='baseline', rescale=True)
print(f"steel supply mix applied to {len(steel_supply)} regions; IT = {steel_supply.get('IT')}")

Pool the trade of the selected commodities — **electricity, steel and aluminium**. MARIO adds a `" supply"` / `" need"` pass-through layer per commodity and stores the observed bilateral trade shares in the supply-block market shares; the technology mix and the trade mix then live in two separate market-share columns, updatable independently. The suffixes match the `NXS2` namespace rows in nxbase. (Steel and aluminium are pooled because they are high-materiality, heterogeneous-footprint traded materials — the clearest goods after electricity on the pooling KPI; aluminium especially is electricity-mix-driven, so where it is smelted matters a lot.)

In [ ]:
traded_commodities = ['Electricity', nxc.STEEL_COMMODITY, nxc.ALUMINIUM_COMMODITY]
db.pool_trade(traded_commodities, supply_suffix=" supply", need_suffix=" need")

**Update trade mixes** in one `trades` scenario, from open sources via the nxbase query API.

- **Pooled** (electricity, steel, aluminium): the mix is rewritten on the supply/need pass-through — electricity from the ENTSO-E scheduled-exchange set (already a share matrix, domestic diagonal included; uncovered destinations forced domestic-only), steel & aluminium from BACI (CEPII) bilateral **flows**.
- **Isard** (chemicals, N-fertiliser, other non-metallic minerals, other non-ferrous metals): high-heterogeneity, heavily-traded goods that are *not* pooled — the import origins are rewritten directly on each base commodity's `u`/`Yc` use columns (`level='Commodity'`). No supply/need commodity is created (zero dimension cost); the corrected mix is embodied in consumers' footprints through the Leontief inverse.

For the BACI commodities, `get_trade_mix` derives each destination's **foreign** sourcing shares (domestic diagonal omitted); `update_trade_mix(rescale=True)` then preserves the base domestic share and rewrites only the imports (BACI has cross-border flows only). HS6→commodity is resolved by the nxbase HS22→CN26→NXS graph. Energy carriers stay aggregated (CN can't separate them; footprint ~uniform by origin).

In [ ]:
scenario = 'trades'
if scenario not in db.scenarios:
    db.clone_scenario('baseline', scenario)

regions = list(db.get_index('Region'))
baci_path = paths.get('baci')  # local CEPII BACI folder; fallback when nxbase has no BACI

# --- Pooled commodities (supply/need pass-through): electricity, steel, aluminium ---

# Electricity — ENTSO-E scheduled-exchange import mix (full matrix incl. the
# domestic diagonal; destinations ENTSO-E does not cover are forced domestic-only).
pooled = db.meta.pooled_trade_map['Electricity']
ele = nxc.get_trade_matrix(
    nxbase_api, year=year, commodity='Electricity',
    source=f"ENTSO-E electricity import mix {year}",
)
ele_dict = {
    dest: (ele[dest].dropna().to_dict() if (dest in ele.columns and ele[dest].sum() > 0) else {dest: 1.0})
    for dest in regions
}
db.update_trade_mix(
    ele_dict, items=pooled['supply'], commodities=pooled['need'],
    scenario=scenario, rescale=True,
)

# Steel & aluminium — BACI bilateral flows (foreign-only; the domestic diagonal is
# omitted so update_trade_mix preserves the base domestic share and rescales the
# foreign origins onto it). BACI from the nxbase API, else the local CEPII file.
for commodity in (nxc.STEEL_COMMODITY, nxc.ALUMINIUM_COMMODITY):
    p = db.meta.pooled_trade_map[commodity]
    mix = nxc.get_trade_mix(nxbase_api, commodity, year=year, regions=regions, baci_path=baci_path)
    db.update_trade_mix(
        mix, items=p['supply'], commodities=p['need'], scenario=scenario, rescale=True,
    )

# --- Isard commodities (NOT pooled): rewrite the import origins directly on the
# base commodity's `u` / `Yc` use columns. Zero dimension cost; consumers' footprints
# embed the corrected mix via Leontief. HS->commodity resolved by the nxbase graph. ---
ISARD_COMMODITIES = [
    "Chemicals nec",
    "N-fertiliser",
    "Other non-metallic mineral products",
    "Other non-ferrous metal products",
]
for commodity in ISARD_COMMODITIES:
    mix = nxc.get_trade_mix(nxbase_api, commodity, year=year, regions=regions, baci_path=baci_path)
    db.update_trade_mix(
        mix, items=[commodity], level='Commodity', scenario=scenario, rescale=True,
    )

print(f"trade mixes applied (scenario '{scenario}'):")
print("  pooled: electricity (ENTSO-E) + steel + aluminium (BACI)")
print(f"  Isard (BACI): {ISARD_COMMODITIES}")

Export v3.1 (electricity + steel supply and trade mixes) — the `trades` scenario.

In [ ]:
out_path = os.path.join(paths['export'], version, str(year))
os.makedirs(out_path, exist_ok=True)
db.to_txt(path = out_path, scenario = scenario)
print('exported ->', out_path)

### v3.0 vs v3.1 — steel footprint

Two effects on *Basic iron and steel*, both country-specific in v3.1:
- **produced** (route/supply mix): domestic production footprint, v3.0 vs v3.1 — more secondary → lower, DRI-heavy regions stay primary;
- **consumed** (import mix): the pooled `… need` commodity footprint in v3.1 vs the domestic *produced* one — how much sourcing steel abroad (BACI) shifts the footprint a market actually embodies. Importers of high-footprint steel go up, importers of clean steel go down.

In [ ]:
import pandas as pd

db_v30 = mario.parse_from_txt(
    os.path.join(paths['export'], 'v3.0', str(year), 'flows'), mode='flows', table='SUT')
gwp = {'Carbon dioxide, fossil (air - Emiss)': 1.0, 'CH4 (air - Emiss)': 29.8, 'N2O (air - Emiss)': 273.0}

def ghg_fp(f):
    f = f.loc[list(gwp), :].T
    return sum(f[s] * w for s, w in gwp.items())

fp30 = ghg_fp(db_v30.f)
fp31 = ghg_fp(db.query('f', scenarios="trades"))

def by_region(fp, item):
    return fp.xs(('Commodity', item), level=('Level', 'Item'))

for label, commodity in [('Steel', nxc.STEEL_COMMODITY), ('Aluminium', nxc.ALUMINIUM_COMMODITY)]:
    need = db.meta.pooled_trade_map[commodity]['need']
    t = pd.DataFrame({
        'produced_v3.0': by_region(fp30, commodity),
        'produced_v3.1': by_region(fp31, commodity),
        'consumed_v3.1': by_region(fp31, need),
    })
    # keep only regions that actually produce the material domestically — otherwise
    # produced == 0 and the ratios are inf (a non-producer consumes 100% imports:
    # its consumed footprint is valid but there is no domestic baseline to compare).
    t = t[t['produced_v3.1'] > 0.01].copy()
    t['route_%'] = 100 * (t['produced_v3.1'] / t['produced_v3.0'] - 1)
    t['import_%'] = 100 * (t['consumed_v3.1'] / t['produced_v3.1'] - 1)
    print(f'\n{label} — GHG footprint per unit (produced vs consumed), domestic producers only '
          f'(route = supply mix, import = trade mix):')
    display(t.round(3).sort_values('import_%').head(15))